# PlantDoc + YOLOv8 - experimentos do TCC
Ative uma GPU em **Runtime > Change runtime type**. Use uma exportação YOLOv8 do PlantDoc que contenha `train`, `valid`, `test` e `data.yaml`.

In [ ]:
!git clone https://github.com/SEU_USUARIO/plantdoc-yolov8-tcc.git
%cd plantdoc-yolov8-tcc
!python -m pip install -e .

## Envio e extração do dataset
Escolha o ZIP exportado pelo Roboflow. Chaves de API não devem ser gravadas no notebook.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
uploaded = files.upload()
zip_name = next(iter(uploaded))
dataset_root = Path('/content/plantdoc')
with zipfile.ZipFile(zip_name) as archive:
    archive.extractall(dataset_root)
data_yaml = next(dataset_root.rglob('data.yaml'))
print(data_yaml)

In [ ]:
!python -m plantdoc_tcc audit --data "{data_yaml}" --expected-classes 27

## Treinamento
Comece por uma estratégia e pela triagem de 100 épocas. O grid integral é grande; promova para 200/300 épocas apenas as melhores configurações na validação. Nunca escolha hiperparâmetros pelo teste.

In [ ]:
!python -m plantdoc_tcc train --data "{data_yaml}" --config configs/experiments.json --strategy baseline --epoch 100 --device 0

Repita com `--strategy class_weighted` e `--strategy focal`. Depois de escolher o melhor modelo somente pela validação, execute a avaliação final uma vez:

In [ ]:
BEST = '/content/plantdoc-yolov8-tcc/runs/plantdoc/SEU_RUN/weights/best.pt'
!python -m plantdoc_tcc evaluate --weights "{BEST}" --data "{data_yaml}" --split test --output artifacts/test_metrics.json